### Étape 1 : Les imports pour créer le réseau de neuronnes

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

### Étape 2 : Définir la classe MicroMLP

In [2]:
class MicroMLP(nn.Module):
    def __init__(self):
        super(MicroMLP, self).__init__()
        input_size = 25  # taille d'entrée
        hidden_size = 1  # taille de couche cachée
        output_size = 10  # taille de sortie
        # Définir les couches ici
        self.couche1 = nn.Linear(input_size, hidden_size)
        self.couche2 = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # Définir le flux de calcul (forward pass)
        x = torch.relu(self.couche1(x))  # Activation ReLU
        x = self.couche2(x)  # Pas d'activation sur la sortie (logits)
        return x

In [3]:
# je crée un instance de mon modele
model = MicroMLP()

# Afficher le nombre total de paramètres
## Parcourir tous les paramètres avec model.parameters()
total_params = sum(p.numel() for p in model.parameters())
print(f"Nombre total de paramètres : {total_params}")

Nombre total de paramètres : 46


### Étape 3 : Charger et préparer MNIST

Maintenant, on va charger le dataset MNIST et sous-échantillonner les images de 28×28 à 5×5 pixels.

In [5]:
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Transformation : convertir en tensor et redimensionner à 5x5
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((5, 5))
])

# Charger MNIST
mnist_train = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

# Créer le DataLoader
train_loader = DataLoader(mnist_train, batch_size=10, shuffle=True)

# Extraire un batch
images, labels = next(iter(train_loader))

# Vérifier les dimensions
print(f"Shape des images : {images.shape}")
print(f"Shape des labels : {labels.shape}") 

Shape des images : torch.Size([10, 1, 5, 5])
Shape des labels : torch.Size([10])


### Étape 4 : Aplatir les images

Le réseau attend des vecteurs de taille 25 en entrée, mais actuellement les images ont la shape [10, 1, 5, 5]. On doit les aplatir en [10, 25].

In [7]:
# Aplatir les images de [10, 1, 5, 5] vers [10, 25]
images_flat = images.view(images.size(0), -1)

# Vérifier la nouvelle shape
print(f"Shape après aplatissement : {images_flat.shape}")

Shape après aplatissement : torch.Size([10, 25])


#### Testons le réseau !

In [8]:
# Forward pass
model = MicroMLP()
output = model(images_flat)

print(f"Shape de la sortie : {output.shape}")

Shape de la sortie : torch.Size([10, 10])


### Étape 5 : Implémenter la fonction HvP

Implémentons le **produit Hessien-vecteur (HvP)** avec l'identité de Pearlmutter. 

#### Rappel : L'identité de Pearlmutter

Pour un vecteur $v$, on veut calculer $Hv$ sans former $H$ :

$$
H(\theta) v = \nabla_\theta \left[ \langle \nabla_\theta L(\theta), v \rangle \right]
$$

**Pour cela, on** : 
1. Calcule le gradient $\nabla_\theta L$ (première rétropropagation)
2. Fait le produit scalaire avec $v$
3. Calcule le gradient de ce produit scalaire (deuxième rétropropagation)

In [9]:
def hessian_vector_product(model, loss_fn, data, labels, v):
    """
    Calcule le produit Hessien-vecteur H*v
    
    Args:
        model: le réseau de neurones
        loss_fn: fonction de perte (ex: nn.CrossEntropyLoss())
        data: batch de données (images aplaties)
        labels: labels correspondants
        v: vecteur avec lequel multiplier le Hessien
    
    Returns:
        Hv: produit Hessien-vecteur
    """
    # Étape 1 : Forward pass
    output = model(data)
    
    # Étape 2 : Calculer la loss
    loss = loss_fn(output, labels)
    
    # Étape 3 : Calculer le gradient (première rétroprop)
    # IMPORTANT: create_graph=True pour garder le graphe pour la 2ème rétroprop
    grads = torch.autograd.grad(
        outputs=loss,
        inputs= model.parameters(),
        create_graph=True
    )
    
    # grads est maintenant un tuple contenant les gradients de tous les paramètres
    return grads  # Pour l'instant, juste pour tester

In [10]:
# Créer un vecteur v aléatoire (on va l'utiliser après)
v = torch.randn(46)  # 46 paramètres dans ton réseau

# Tester la fonction (pour l'instant elle retourne juste grads)
model = MicroMLP()
loss_fn = torch.nn.CrossEntropyLoss()

grads = hessian_vector_product(model, loss_fn, images_flat, labels, v)

# Afficher combien de gradients on a et leur forme
print(f"Nombre de tenseurs de gradients : {len(grads)}")
for i, g in enumerate(grads):
    print(f"Gradient {i} : shape = {g.shape}")

Nombre de tenseurs de gradients : 4
Gradient 0 : shape = torch.Size([1, 25])
Gradient 1 : shape = torch.Size([1])
Gradient 2 : shape = torch.Size([10, 1])
Gradient 3 : shape = torch.Size([10])


On a bien 4 tenseurs de gradients correspondant aux 4 groupes de paramètres de ton réseau :

- Gradient 0 : Poids de la couche 1 (25 → 1) : shape [1, 25] → 25 paramètres

- Gradient 1 : Biais de la couche 1 : shape [1] → 1 paramètre

- Gradient 2 : Poids de la couche 2 (1 → 10) : shape [10, 1] → 10 paramètres

- Gradient 3 : Biais de la couche 2 : shape [10] → 10 paramètres

Total : 25 + 1 + 10 + 10 = 46 paramètres 

In [11]:
# Étape 4 : Aplatir et concaténer tous les gradients
grad_vec = torch.cat([g.view(-1) for g in grads])

# Produit scalaire <grad, v>
dot_product = torch.dot(grad_vec, v)

print(f"Produit scalaire : {dot_product}")
print(f"Type : {type(dot_product)}")  # Devrait être un scalaire (tensor 0D)

Produit scalaire : 0.12159258127212524
Type : <class 'torch.Tensor'>


In [12]:
# Étape 5 : Gradient du produit scalaire (deuxième rétroprop)
Hv_grads = torch.autograd.grad(
    outputs=dot_product,
    inputs=model.parameters(),
    create_graph=False  # Plus besoin du graphe après
)

# Aplatir et concaténer pour obtenir Hv comme un vecteur
Hv = torch.cat([g.contiguous().view(-1) for g in Hv_grads])

print(f"Shape de Hv : {Hv.shape}")  # Devrait être [46]
print(f"Premiers éléments de Hv : {Hv[:5]}")

Shape de Hv : torch.Size([46])
Premiers éléments de Hv : tensor([ 0.0000, -0.0004,  0.0005,  0.0074,  0.0002])


In [13]:
def hessian_vector_product(model, loss_fn, data, labels, v):
    """
    Calcule le produit Hessien-vecteur H*v
    
    Args:
        model: le réseau de neurones
        loss_fn: fonction de perte (ex: nn.CrossEntropyLoss())
        data: batch de données (images aplaties)
        labels: labels correspondants
        v: vecteur avec lequel multiplier le Hessien
    
    Returns:
        Hv: produit Hessien-vecteur (vecteur de même taille que v)
    """
    # Étape 1 : Forward pass
    output = model(data)
    
    # Étape 2 : Calculer la loss
    loss = loss_fn(output, labels)
    
    # Étape 3 : Calculer le gradient (première rétroprop)
    grads = torch.autograd.grad(
        outputs=loss,
        inputs=model.parameters(),
        create_graph=True
    )
    
    # Étape 4 : Produit scalaire <grad, v>
    grad_vec = torch.cat([g.view(-1) for g in grads])
    dot_product = torch.dot(grad_vec, v)
    
    # Étape 5 : Gradient du produit scalaire (deuxième rétroprop)
    Hv_grads = torch.autograd.grad(
        outputs=dot_product,
        inputs=model.parameters(),
        create_graph=False
    )
    
    # Aplatir et concaténer pour obtenir Hv
    Hv = torch.cat([g.contiguous().view(-1) for g in Hv_grads])
    
    return Hv

In [14]:
# Test
model = MicroMLP()
loss_fn = torch.nn.CrossEntropyLoss()
v = torch.randn(46)

Hv = hessian_vector_product(model, loss_fn, images_flat, labels, v)
print(f"Shape de Hv : {Hv.shape}")

Shape de Hv : torch.Size([46])


### Partie 1.3 : Validation Cruciale

Maintenant on passe à l'étape la plus importante du projet : valider que le HvP est correct en le comparant avec le Hessien explicite calculé manuellement.

#### Stratégie de validation

On va :
1. **Calculer explicitement** la matrice Hessienne $H$ (46×46) en calculant chaque colonne avec des gradients
2. **Calculer** $H \cdot v$ par multiplication matricielle
3. **Comparer** avec le résultat de ta fonction `hessian_vector_product`

#### Étape 1 : Calculer le Hessien explicite

Voici le principe : chaque **colonne $j$** du Hessien est $\nabla^2 L \cdot e_j$ où $e_j$ est le vecteur de base (0, 0, ..., 1, ..., 0) avec un 1 en position $j$.

In [15]:
def compute_hessian_explicit(model, loss_fn, data, labels):
    """
    Calcule explicitement la matrice Hessienne H (TRÈS COÛTEUX !)
    Ne fonctionne que pour des petits réseaux.
    """
    n_params = sum(p.numel() for p in model.parameters())
    H = torch.zeros(n_params, n_params)
    
    # Pour chaque colonne j
    for j in range(n_params):
        # Créer le vecteur de base e_j
        e_j = torch.zeros(n_params)
        e_j[j] = 1.0
        
        # Calculer H * e_j avec ta fonction HvP
        H[:, j] = hessian_vector_product(model, loss_fn, data, labels, e_j)
    
    return H

#### Test de validation

On va :
1. Calculer le Hessien explicite $H$
2. Choisir un vecteur aléatoire $v$
3. Calculer $Hv$ de **deux façons différentes** :
   - Avec ta fonction `hessian_vector_product()`
   - Avec la multiplication matricielle `H @ v`
4. Comparer les deux résultats

In [16]:
# Créer un modèle frais et un petit batch
model = MicroMLP()
loss_fn = torch.nn.CrossEntropyLoss()

# Utiliser seulement 5 images pour aller plus vite
data_small = images_flat[:5]
labels_small = labels[:5]

print("Calcul du Hessien explicite (ça peut prendre 10-20 secondes)...")
H_explicit = compute_hessian_explicit(model, loss_fn, data_small, labels_small)
print(f"Shape du Hessien : {H_explicit.shape}")

# Vecteur de test aléatoire
v_test = torch.randn(46)

# Méthode 1 : Avec ta fonction HvP
Hv_fast = hessian_vector_product(model, loss_fn, data_small, labels_small, v_test)

# Méthode 2 : Multiplication matricielle explicite
Hv_explicit = H_explicit @ v_test

# Comparer les deux
difference = torch.norm(Hv_fast - Hv_explicit).item()
print(f"\n=== VALIDATION ===")
print(f"Norme de la différence : {difference}")
print(f"Différence relative : {difference / torch.norm(Hv_explicit).item():.2e}")

# Afficher quelques valeurs pour comparaison visuelle
print(f"\nPremiers éléments de Hv_fast     : {Hv_fast[:5]}")
print(f"Premiers éléments de Hv_explicit : {Hv_explicit[:5]}")

# Test de validation
if difference < 1e-5:
    print("\n✅ VALIDATION RÉUSSIE ! Ton HvP est correct !")
else:
    print(f"\n❌ ATTENTION : Différence trop grande ({difference})")

Calcul du Hessien explicite (ça peut prendre 10-20 secondes)...
Shape du Hessien : torch.Size([46, 46])

=== VALIDATION ===
Norme de la différence : 1.2842406249546912e-07
Différence relative : 2.17e-07

Premiers éléments de Hv_fast     : tensor([ 0.0000,  0.0302,  0.1045, -0.0169, -0.0024])
Premiers éléments de Hv_explicit : tensor([ 0.0000,  0.0302,  0.1045, -0.0169, -0.0024])

✅ VALIDATION RÉUSSIE ! Ton HvP est correct !


La fonction `hessian_vector_product` est **parfaitement correcte** ! 

- **Norme de la différence** : `1.28e-07` (quasi-nulle, juste des erreurs d'arrondi numérique)
- **Différence relative** : `2.17e-07` (0.000022% d'erreur !)
- Les valeurs sont **identiques** à la précision machine près ✅


#### Récapitulatif de la Partie 1

On a réussi à :
1. ✅ Créer un micro-réseau MLP (25 → 1 → 10) avec 46 paramètres
2. ✅ Charger et préparer MNIST (sous-échantillonné à 5×5)
3. ✅ Implémenter le produit Hessien-vecteur avec l'identité de Pearlmutter
4. ✅ **Valider** l'implémentation contre le Hessien explicite


## Partie 2 : Lanczos et Analyse Spectrale

Maintenant on passe à la **partie 2** : implémenter l'**algorithme de Lanczos** pour trouver les valeurs propres dominantes du Hessien. 

### Rappel : L'algorithme de Lanczos

Lanczos construit une **base orthonormée** de l'espace de Krylov et produit une **matrice tridiagonale** $T_m$ dont les valeurs propres approximent celles du Hessien.